# SchematicSVG demo
Run the cells; each schematic is stored in the notebook as inline SVG.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # use the package from this repository
using SchematicSVG

## CircuiTikZ
Paste CircuiTikZ code (or the output of your generator) and it displays directly.

In [ ]:
tikz"""
\begin{circuitikz}
  \draw (0,0) node[ground]{} to[V, l=$v_{in}$] (0,2) to[C=$C_1$] (2,2) coordinate (B)
        to[R=$R_{B1}$, *-] (2,5) node[vcc]{$V_{CC}$};
  \draw (B) to[R=$R_{B2}$] (2,0) node[ground]{};
  \draw (B) -- (3.5,2) node[npn, anchor=B](Q1){$Q_1$};
  \draw (Q1.C) to[R=$R_C$] ++(0,2.23) node[vcc]{$V_{CC}$};
  \draw (Q1.E) to[R=$R_E$] ++(0,-1.23) node[ground]{};
  \draw (Q1.C) to[short, *-o] ++(1.5,0) node[right]{$v_o$};
\end{circuitikz}
"""

In [ ]:
inverting = raw"""
\begin{circuitikz}
  \draw (0,0) node[op amp](OA){};
  \draw (OA.-) -- ++(-0.5,0) coordinate(m) to[R=$R_1$] ++(-2.5,0) node[left]{$v_{in}$};
  \draw (m) |- ++(0,1.5) coordinate(t) to[R=$R_2$] (t -| OA.out) -- (OA.out);
  \draw (OA.+) -- ++(-0.5,0) node[ground]{};
  \draw (OA.out) to[short,-o] ++(1,0) node[right]{$v_{out}$};
\end{circuitikz}
"""
tikz_schematic(inverting; elements=Dict("R1" => 10e3, "R2" => 100e3))

## Inverting amplifier from ASCII art

In [ ]:
schem"""
            +--gnd
            |
            OA1-------+---vout
            |         |
  vin--R1---+---R2----+
"""

## Sources, a BJT and supply rails, with custom labels

In [ ]:
art = """
    vdd             vdd
     |               |
     I1              RC
     |               |
     +-----+         +--vo
     |     |         |
     D1v   C2  in--|Q1
     |     |         |
    gnd   gnd        G1
                     |
                    vss
"""
ascii_schematic(art; labels=Dict("RC" => raw"$R_C = 2\,k\Omega$", "G1" => raw"$g_m v_{be}$"))

## CMOS inverters with a crossing

In [ ]:
schem"""
           vdd           vdd
            |             |
      +---|P1     +------P2|--+
      |     |     |       |   |
  in--*     *-----)-------*   |
      |     |     |       |   |
      +---|N1     |      N2|--+
            |     |       |
           gnd    |      gnd
                  |
      vb---<D1----+---E1----out
"""

## Annotations from netlist elements
Any object with a `name` and a `value` or `model` field works; here, NamedTuples.

In [ ]:
circuit = [
    (name="RB1", value=47e3), (name="RB2", value=10e3), (name="RC", value=4.7e3),
    (name="RE", value=1e3), (name="C1", value=10e-6), (name="C2", value=2.2e-6),
    (name="Q1", model="2N3904"),
]
amp = """
         vdd               vdd
          |                 |
          RB1               RC
          |                 |
          |                 +---C2---vo
          |                 |
   in--C1-+--@vb-----------|Q1
          |                 |
          RB2               RE
          |                 |
         gnd               gnd
"""
ascii_schematic(amp; elements=circuit, nodes=Dict("vb" => 1.9, "vo" => raw"$v_o$"))

In [ ]:
ascii_schematic(amp; elements=circuit, style=:inline)

## Dependent sources with values

In [ ]:
ascii_schematic("""
  +---V1---+---E1---+---G1---+
  |        |        |        |
 gnd       H1       F1      gnd
           |        |
          gnd      gnd
"""; elements=Dict("V1" => 5, "E1" => 10, "G1" => 2e-3, "H1" => 1e3, "F1" => 50))

## Placed primitives (e.g. from an existing CircuiTikZ layout)

In [ ]:
Schematic([
    Bipole(:V, Pt(0, 1), Pt(0, 0), raw"$v_{in}$"), Ground(Pt(0, 0)),
    Wire(Pt(0, 1), PinRef("Q1", :B)),
    Transistor(:npn, "Q1", Pt(1, 1)),
    Bipole(:R, Pt(1, 2.5), Pt(1, 1.5), raw"$R_C$"), Wire(Pt(1, 1.5), PinRef("Q1", :C)),
    Supply(Pt(1, 2.5), "V_{CC}", true),
    Wire(PinRef("Q1", :E), Pt(1, 0)), Ground(Pt(1, 0)),
])

## Netlists
`spice_netlist` works on its own; with Microelectronics.jl loaded, `to_microelectronics` returns element objects for `build_netlist`.

In [ ]:
cmos = """
             vdd
              |
        +---|P1
        |     |
  in----*     +--@out--+
        |     |        |
        +---|N1        CL
        |     |        |
        V1   gnd      gnd
        |
       gnd
"""
print(spice_netlist(ascii_schematic(cmos); elements=Dict("V1" => 0, "CL" => 1e-12),
                    supplies=Dict("vdd" => 5)))

In [ ]:
# using Microelectronics
# ckt = to_microelectronics(ascii_schematic(cmos); elements=Dict("V1" => 5u"V", "CL" => 1u"pF"),
#                           supplies=Dict("vdd" => 5u"V"))
# build_netlist(; ckt...); simulate(:op)